# Optimizar Ensemble Medium


In [18]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

DATA = Path("data")
MODELS = Path("models")
ENSEMBLE_DIR = MODELS / "medium_ensemble"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


In [19]:
target_scaler = joblib.load(MODELS / "target_scaler_v2.joblib")

Y_MEAN = float(target_scaler["mean"])
Y_STD = float(target_scaler["std"])

print("X_val:", X_val.shape)
print("y_val:", y_val_usd.shape)
print(f"Target mean: ${Y_MEAN:,.2f}")
print(f"Target std:  ${Y_STD:,.2f}")


X_val: (234, 278)
y_val: (234,)
Target mean: $181,637.14
Target std:  $78,015.42


In [20]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout=0.1):
        super().__init__()
        layers = []
        prev = input_dim

        for h in hidden_layers:
            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


In [21]:
checkpoint_files = sorted(ENSEMBLE_DIR.glob("*.pt"))

if len(checkpoint_files) != 5:
    raise FileNotFoundError(
        f"Se esperaban 5 checkpoints en {ENSEMBLE_DIR}, pero se encontraron {len(checkpoint_files)}: "
        f"{[p.name for p in checkpoint_files]}"
    )

print("Checkpoints encontrados:")
for p in checkpoint_files:
    print(" -", p.name)


Checkpoints encontrados:
 - medium_seed_123.pt
 - medium_seed_2026.pt
 - medium_seed_42.pt
 - medium_seed_456.pt
 - medium_seed_789.pt


In [22]:
def predict_checkpoint(path, X):
    ckpt = torch.load(path, map_location=DEVICE)

    hidden = ckpt.get("hidden_layers", [128, 64])
    hp = ckpt.get("hyperparams", {})
    dropout = hp.get("dropout", ckpt.get("dropout", 0.1))

    model = MLPRegressor(
        input_dim=X.shape[1],
        hidden_layers=hidden,
        dropout=dropout
    ).to(DEVICE)

    state = ckpt["model_state_dict"]
    model.load_state_dict(state)
    model.eval()

    with torch.no_grad():
        pred_scaled = model(
            torch.tensor(
                X,
                dtype=torch.float32,
                device=DEVICE
            )
        ).cpu().numpy()

    # Regresar de target estandarizado a dólares
    pred_usd = pred_scaled * Y_STD + Y_MEAN

    return pred_usd


In [23]:
val_predictions = []
individual_rows = []

for path in checkpoint_files:
    pred = predict_checkpoint(path, X_val)
    val_predictions.append(pred)

    rmse = np.sqrt(np.mean((y_val_usd - pred) ** 2))

    individual_rows.append({
        "modelo": path.name,
        "RMSE_val": rmse
    })

individual_df = pd.DataFrame(individual_rows).sort_values("RMSE_val")
display(individual_df)

val_predictions = np.vstack(val_predictions)


,modelo,RMSE_val
3,medium_seed_456.pt,23171.968750
2,medium_seed_42.pt,23768.275391
0,medium_seed_123.pt,24142.681641
1,medium_seed_2026.pt,24262.572266
4,medium_seed_789.pt,25921.699219


In [ ]:
rows = []

for k in range(1, len(checkpoint_files) + 1):
    for idxs in combinations(range(len(checkpoint_files)), k):

        pred = val_predictions[list(idxs)].mean(axis=0)
        rmse = np.sqrt(np.mean((y_val_usd - pred) ** 2))

        rows.append({
            "n_modelos": k,
            "indices": idxs,
            "modelos": [checkpoint_files[i].name for i in idxs],
            "RMSE_val": rmse
        })

results = pd.DataFrame(rows).sort_values("RMSE_val").reset_index(drop=True)

display(results.head(15))


,n_modelos,indices,modelos,RMSE_val
0,2,"(2, 3)","[medium_seed_42.pt, medium_seed_456.pt]",22680.339844
1,3,"(1, 2, 3)","[medium_seed_2026.pt, medium_seed_42.pt, mediu...",22760.414062
2,3,"(0, 2, 3)","[medium_seed_123.pt, medium_seed_42.pt, medium...",22825.490234
3,4,"(0, 1, 2, 3)","[medium_seed_123.pt, medium_seed_2026.pt, medi...",22848.531250
4,2,"(1, 3)","[medium_seed_2026.pt, medium_seed_456.pt]",22874.455078
5,3,"(0, 1, 3)","[medium_seed_123.pt, medium_seed_2026.pt, medi...",22974.929688
6,2,"(0, 3)","[medium_seed_123.pt, medium_seed_456.pt]",23105.628906
7,4,"(1, 2, 3, 4)","[medium_seed_2026.pt, medium_seed_42.pt, mediu...",23132.001953
8,5,"(0, 1, 2, 3, 4)","[medium_seed_123.pt, medium_seed_2026.pt, medi...",23133.458984
9,1,"(3,)",[medium_seed_456.pt],23171.968750


In [25]:
best = results.iloc[0]
best_idxs = list(best["indices"])

all_five_pred = val_predictions.mean(axis=0)
rmse_all_five = np.sqrt(np.mean((y_val_usd - all_five_pred) ** 2))

print("Mejor combinación:")
for i in best_idxs:
    print(" -", checkpoint_files[i].name)

print(f"\nNúmero de modelos: {len(best_idxs)}")
print(f"RMSE validación mejor combinación: ${best['RMSE_val']:,.2f}")
print(f"RMSE ensemble de 5:              ${rmse_all_five:,.2f}")
print(f"Mejora vs ensemble de 5:         ${rmse_all_five - best['RMSE_val']:,.2f}")


Mejor combinación:
 - medium_seed_42.pt
 - medium_seed_456.pt

Número de modelos: 2
RMSE validación mejor combinación: $22,680.34
RMSE ensemble de 5:              $23,133.46
Mejora vs ensemble de 5:         $453.12


## Generar predictions.csv

Esta sección usa el mismo preprocesador original y conserva los `Id` del archivo de test.


In [26]:
TEST_PATH = DATA / "test_features-1.csv"
EXPECTED_PATH = Path("expected_output.csv")

test_df = pd.read_csv(TEST_PATH)
test_ids = test_df["Id"].copy()

X_test_raw = test_df.drop(columns=["SalePrice"], errors="ignore").copy()
X_test_raw = X_test_raw.drop(columns=["Id"], errors="ignore")

preprocessor = joblib.load(MODELS / "preprocessor_v2.joblib")
X_test = preprocessor.transform(X_test_raw)

if hasattr(X_test, "toarray"):
    X_test = X_test.toarray()

X_test = np.asarray(X_test, dtype=np.float32)

print("Test procesado:", X_test.shape)


Test procesado: (292, 278)


In [ ]:
test_predictions = []

for i in best_idxs:
    pred = predict_checkpoint(checkpoint_files[i], X_test)
    test_predictions.append(pred)

pred_final = np.mean(np.vstack(test_predictions), axis=0)

predictions = pd.DataFrame({
    "Id": test_ids.to_numpy(),
    "Prediction": pred_final
})

assert predictions.columns.tolist() == ["Id", "Prediction"]
assert len(predictions) == len(test_df)
assert predictions["Prediction"].notna().all()

if EXPECTED_PATH.exists():
    expected = pd.read_csv(EXPECTED_PATH)

    assert expected.columns.tolist() == ["Id", "Prediction"], (
        f"expected_output.csv tiene columnas {expected.columns.tolist()}"
    )
    assert len(expected) == len(predictions), (
        f"Filas distintas: expected={len(expected)}, predictions={len(predictions)}"
    )
    assert np.array_equal(expected["Id"].to_numpy(), predictions["Id"].to_numpy()), (
        "Los Id no coinciden con expected_output.csv"
    )

predictions.to_csv("predictions.csv", index=False)

print("✓ predictions.csv generado")
print("Columnas:", predictions.columns.tolist())
print("Filas:", len(predictions))
display(predictions.head())


✓ predictions.csv generado
Columnas: ['Id', 'Prediction']
Filas: 292


,Id,Prediction
0,893,149851.12500
1,1106,342870.21875
2,414,101926.56250
3,523,168969.31250
4,1037,363050.56250
